# PERMANOVA / PERMDISP screen (scikit-bio): design, phenotypes, batch

A screen of many metadata variables for 16S (ASV, bacteria) and ITS1 (OTU97, fungi).
The reported tests stay in R (`scripts/10a_16S_permanova.R`, `10b_ITS1_permanova.R`:
`vegan::adonis2` with tree blocking and tree-level models). This notebook asks
which variables are worth a closer look, and whether batch columns explain community
differences.

**Distances** (10% prevalence filter, as 08a/08b):
- `aitchison_CLR` (primary): multiplicative zero replacement (`skbio multi_replace`),
  CLR, Euclidean. R uses `zCompositions::cmultRepl` (CZM), so values differ slightly.
- `aitchison_rCLR` (sensitivity): rCLR (zeros left out of the geometric mean), Euclidean.
- `bray_curtis`, `jaccard`: mean over repeated rarefactions (as `vegan::avgdist`).

**Tests.** scikit-bio `permanova` and `permdisp` are one-way tests with free
permutations: no covariates, no strata. Flowers are nested in trees in farms, so free
permutations overstate significance for treatment and farm-level variables. Each test
is therefore run twice:
- *free*: scikit-bio, labels shuffled across all flowers;
- *restricted*: labels shuffled within strata (tree, farm, or farm x treatment), by a
  short helper below. Its observed F is checked against scikit-bio.

PERMDISP uses distances to group **centroids** (`test='centroid'`), so the same
distances can be re-used under restricted permutations. Farm-level variables
(farm, management type, collection day) cannot be tested at flower level and are
shown as descriptive R² only; their test is the tree-level model in R 10a/10b.

Variable blocks: **design** (treatment, farm), **phenotype** (microscopy; measured on
a subset of flowers, given n), **batch** (extraction, PCR, Qubit, primers, PCR redo).
Batch layout and the questions asked here: `docs/analysis_log.md`, "Batch effects check".

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
import biom
from scipy.spatial.distance import pdist, squareform
from scipy.stats.contingency import association
from statsmodels.stats.multitest import multipletests
from skbio import DistanceMatrix
from skbio.diversity import beta_diversity
from skbio.stats import subsample_counts
from skbio.stats.composition import multi_replace, clr
from skbio.stats.distance import permanova, permdisp
from skbio.stats.ordination import pcoa
# rCLR: scikit-bio 0.7.2 has no rclr; deicode (as in notebook 02a) returns NaN for
# zeros, which are set to 0 after the transform (as vegan robust.aitchison)
from deicode.preprocessing import rclr

warnings.filterwarnings('ignore')

In [ ]:
# Data version: v1 | v2 (default v2), as CFM_VERSION in scripts/utils_paths.R.
# Sample set: main = 288 flowers sampled by the standard protocol (the final set),
# with_deviations = all 294. Set from the environment for a batch run, or edit the
# default here in Jupyter. The set goes into every figure and table name (_288 / _294).
CFM_VERSION = os.environ.get('CFM_VERSION', 'v2')
SAMPLE_SET  = os.environ.get('CFM_SAMPLE_SET', 'main')
assert CFM_VERSION in ('v1', 'v2'), CFM_VERSION
assert SAMPLE_SET in ('main', 'with_deviations'), SAMPLE_SET
BIOM_SUFFIX = '' if SAMPLE_SET == 'main' else '_with_deviations'
FIG_SUFFIX  = '_288' if SAMPLE_SET == 'main' else '_294'

ROOT     = Path('..').resolve()
BIOM_DIR = ROOT / 'results' / CFM_VERSION / 'biom'
DATA_DIR = ROOT / 'data'
TAB_DIR  = ROOT / 'results' / CFM_VERSION / 'tables'
FIG_DIR  = ROOT / 'results' / CFM_VERSION / 'figures' / 'permanova'
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data {CFM_VERSION}, sample set {SAMPLE_SET} ({FIG_SUFFIX[1:]} flowers) <- {BIOM_DIR}')

# Figures: each one is saved as a PNG at 300 dpi (submission format; change dpi here).
# All figures of the notebook also go into one PDF for viewing, written by the last
# cell. FIGURES is keyed by name, so re-running a cell replaces its page.
FIGURES = {}

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f'{name}{FIG_SUFFIX}.png', dpi=300, bbox_inches='tight')
    FIGURES[name] = fig

In [ ]:
# Permutations per test. CFM_N_PERM=99 gives a quick test run of the whole notebook;
# results for the paper use 999 (as R 10a/10b).
N_PERM = int(os.environ.get('CFM_N_PERM', 999))
ALPHA  = 0.05   # one significance threshold for all figures
SEED   = 42

PREV_THRESHOLD = 0.10                            # features in >= 10% of flowers (08a/08b)
RAREFY_DEPTH   = {'16S': 2000, 'ITS1': 15000}    # reads per flower for BC and Jaccard (08a/08b)
RAREFY_ITER    = 100                             # rarefactions averaged (08a/08b)
PI_THRESH      = 40                              # pi_clsm cut-off for high_poll (14a)
MIN_GROUP_N    = 3                               # groups with fewer flowers are dropped from a test

# The results tables take ~1.3 h at 999 permutations (both markers, all metrics).
# They are saved and read back on the next run if made with the same N_PERM;
# set True to recompute.
FORCE_RERUN = False

# biom file of each marker; the marker suffix of its metadata columns
MARKERS    = {'16S': 'cfm_16s_bacteria_biosamples', 'ITS1': 'cfm_its1_otu97_fungi_biosamples'}
MARKER_COL = {'16S': '16s', 'ITS1': 'its1'}
METRICS    = ['aitchison_CLR', 'aitchison_rCLR', 'bray_curtis', 'jaccard']

# Farm order: inside_forest -> full_sun deforestation gradient
FARM_LEVELS = ['ib', 'vr', 'sa', 'kk', 'mt', 'vi', 'yb']

## Metadata and derived variables

Groupings are copied from the R scripts:
- `farm_treatment`: farm x treatment (01c);
- `pollination_class`: `pi_clsm` 0 / 1-39 / >= 40 = unpollinated / low_poll / high_poll (14a);
- `fungal_col_0v5`: `fungal_colonization_score` 0 vs 5 (15a);
- `pcr_redo_<marker>_any`: at least one PCR replicate redone (01c).

In [ ]:
meta = pd.read_csv(
    DATA_DIR / 'sample_metadata_with_microscopy.txt',
    sep='\t', index_col=0, dtype=str,
)
# Remove the QIIME 2 types row
meta = meta.drop(index='#q2:types', errors='ignore')
meta = meta[meta['sample_type'].isin(['bagged_flower', 'unbagged_flower'])].copy()

meta['farm_treatment'] = meta['farm_id'] + ':' + meta['sample_type']

# pi_clsm is a whole number of pollen grains; the bins give 0 | 1-39 | >= 40
meta['pollination_class'] = pd.cut(
    pd.to_numeric(meta['pi_clsm']),
    bins=[-0.5, 0.5, PI_THRESH - 0.5, np.inf],
    labels=['unpollinated', 'low_poll', 'high_poll'],
).astype(object)

# Scores 1-4 are left out of this contrast (NaN)
meta['fungal_col_0v5'] = meta['fungal_colonization_score'].map({'0': 'score_0', '5': 'score_5'})

for m in MARKER_COL.values():
    meta[f'pcr_redo_{m}_any'] = np.where(pd.to_numeric(meta[f'pcr_redo_{m}']) >= 1,
                                         'redo', 'no_redo')
print(f'Metadata: {meta.shape[0]} flowers (before matching the biom)')

## Distance matrices

One `DistanceMatrix` per marker and metric, and one PCoA of each (used by PERMDISP and
the figures). For the Aitchison metrics the PCoA equals a PCA of the transformed data.
Bray-Curtis and Jaccard drop flowers below the rarefaction depth, so their n can be
smaller.

In [ ]:
def filter_counts(counts_df, min_prevalence=0.0, min_total=0):
    """Keep features meeting minimum prevalence and total-count thresholds.
    Copied from notebook 02a; counts_df is features x samples."""
    n_samples  = counts_df.shape[1]
    prevalence = (counts_df > 0).sum(axis=1) / n_samples
    total      = counts_df.sum(axis=1)
    keep       = (prevalence >= min_prevalence) & (total >= min_total)
    return counts_df[keep]


def prepare_clr(counts_df):
    """CLR-transform a features x samples count matrix (copied from notebook 02a).

    multi_replace applies TSS normalisation then fills zeros with
    delta = 1/N^2 (Martin-Fernandez multiplicative replacement).
    Returns a samples x features DataFrame.
    """
    comp = multi_replace(counts_df.T.values)
    return pd.DataFrame(clr(comp), index=counts_df.columns, columns=counts_df.index)


def prepare_rclr(counts_df):
    """rCLR of a features x samples count matrix: log ratio to the geometric mean of
    the non-zero counts of each sample. Zeros are set to 0 after the transform, as in
    vegan::vegdist(method = 'robust.aitchison'). Returns samples x features."""
    x = np.nan_to_num(rclr(counts_df.T.values), nan=0.0)
    return pd.DataFrame(x, index=counts_df.columns, columns=counts_df.index)


def euclidean_dm(df):
    """Euclidean DistanceMatrix between the rows (samples) of df."""
    return DistanceMatrix(squareform(pdist(df.values)), ids=df.index)


def rarefied_dm(counts_df, metric, depth, iterations, seed):
    """Mean distance over `iterations` rarefactions to `depth` reads, as
    vegan::avgdist. Flowers with fewer reads than depth are dropped.
    Jaccard is computed on presence/absence (scipy's jaccard on counts would
    compare the count values, not presence)."""
    keep = counts_df.columns[counts_df.sum() >= depth]
    x    = counts_df[keep].T.values.astype(int)
    rng  = np.random.default_rng(seed)
    total = np.zeros((len(keep), len(keep)))
    for _ in range(iterations):
        sub = np.vstack([subsample_counts(row, depth, seed=rng) for row in x])
        if metric == 'jaccard':
            sub = (sub > 0).astype(int)
        total += beta_diversity(metric, sub, ids=keep).data
    return DistanceMatrix(total / iterations, ids=keep)

In [ ]:
clr_data = {}   # samples x features CLR per marker; used by the batch 2024-09-20 check
dist     = {}   # dist[marker][metric] -> skbio DistanceMatrix
ords     = {}   # ords[marker][metric] -> skbio OrdinationResults (PCoA), for the figures

for marker, stem in MARKERS.items():
    tbl    = biom.load_table(str(BIOM_DIR / f'{stem}{BIOM_SUFFIX}_feature-table.biom'))
    counts = tbl.to_dataframe(dense=True)   # features x samples
    assert set(counts.columns) <= set(meta.index), 'biom holds samples that are not flowers'
    counts_f = filter_counts(counts, PREV_THRESHOLD)
    print(f'{marker}: {counts.shape[0]} -> {counts_f.shape[0]} features after prevalence '
          f'>= {PREV_THRESHOLD:.0%}; {counts_f.shape[1]} flowers')

    clr_data[marker] = prepare_clr(counts_f)
    dist[marker] = {
        'aitchison_CLR':  euclidean_dm(clr_data[marker]),
        'aitchison_rCLR': euclidean_dm(prepare_rclr(counts_f)),
        'bray_curtis':    rarefied_dm(counts_f, 'braycurtis', RAREFY_DEPTH[marker], RAREFY_ITER, SEED),
        'jaccard':        rarefied_dm(counts_f, 'jaccard',    RAREFY_DEPTH[marker], RAREFY_ITER, SEED),
    }
    ords[marker] = {metric: pcoa(dm) for metric, dm in dist[marker].items()}
    for metric in ['bray_curtis', 'jaccard']:
        n_out = counts_f.shape[1] - dist[marker][metric].shape[0]
        print(f'  {metric}: rarefied to {RAREFY_DEPTH[marker]} reads x {RAREFY_ITER}; '
              f'{n_out} flowers below depth dropped')

## Test helpers

scikit-bio gives the observed statistics and the free-permutation p-values. The
restricted p-values come from `restricted_p` below: the same pseudo-F, with the
labels shuffled only within strata. No maintained Python package offers
permutations within strata, and calling scikit-bio once per permutation takes about
30 s per test, so the statistic is computed here in vectorised form:

pseudo-F = [(SS_T - SS_W) / (g - 1)] / [SS_W / (n - g)], where SS_T is the sum of
squared distances / n and SS_W is the same within each group, divided by the group size.

For PERMDISP the distances of the flowers to their group centroid (in the PCoA
coordinates, as scikit-bio) are computed once; the test is then a one-way ANOVA of these
distances. The one-way ANOVA F equals the pseudo-F of the distance matrix |z_i - z_j|,
so the same function serves both (as R 10a: `adonis2(dist(bd$distances) ~ ...)`).

In [ ]:
def pseudo_f(d2, labels):
    """PERMANOVA pseudo-F from a matrix of squared distances d2 and integer group
    labels 0..g-1. Group sums of d2 are taken with a one-hot matrix (G^T d2 G)."""
    n, g = len(labels), labels.max() + 1
    onehot = np.eye(g)[labels]                       # n x g
    sizes  = onehot.sum(axis=0)
    ss_t   = d2.sum() / (2 * n)                      # each pair is counted twice in d2
    ss_w   = (np.diag(onehot.T @ d2 @ onehot) / (2 * sizes)).sum()
    return ((ss_t - ss_w) / (g - 1)) / (ss_w / (n - g))


def shuffle_within(labels, strata, rng):
    """Permute labels within each stratum (all flowers one stratum if strata is None)."""
    if strata is None:
        return rng.permutation(labels)
    out = labels.copy()
    for s in np.unique(strata):
        idx = np.flatnonzero(strata == s)
        out[idx] = rng.permutation(labels[idx])
    return out


def restricted_p(d2, labels, strata, f_obs, n_perm, seed):
    """Permutation p-value of pseudo-F with labels shuffled within strata."""
    rng    = np.random.default_rng(seed)
    f_perm = np.array([pseudo_f(d2, shuffle_within(labels, strata, rng))
                       for _ in range(n_perm)])
    return (np.sum(f_perm >= f_obs - 1e-12) + 1) / (n_perm + 1)


def centroid_distances(ordination, labels):
    """Distance of each flower to the centroid of its group in the PCoA coordinates
    (all axes, as skbio permdisp with test='centroid')."""
    coords = ordination.samples.values
    out    = np.empty(len(labels))
    for lab in np.unique(labels):
        idx = labels == lab
        out[idx] = np.linalg.norm(coords[idx] - coords[idx].mean(axis=0), axis=1)
    return out


def r2_from_f(f, n, g):
    """R2 of a one-way PERMANOVA from its pseudo-F: SS_A / SS_T."""
    return f * (g - 1) / (f * (g - 1) + (n - g))


def cached(path, compute):
    """Read a saved results table, or compute it with compute() and save it.
    A table made with another N_PERM (a quick test run) is recomputed."""
    if path.exists() and not FORCE_RERUN:
        df = pd.read_csv(path)
        if (df['n_perm'] == N_PERM).all():
            print(f'Read {path.name} ({len(df)} rows); FORCE_RERUN = True to recompute')
            return df
    df = compute()
    df.to_csv(path, index=False)
    print(f'Saved {path}')
    return df

In [ ]:
def run_test(dm, grouping, strata=None):
    """One-way PERMANOVA and PERMDISP of `grouping` (pd.Series indexed by sample id,
    NaN = not measured). Flowers missing from dm or in groups smaller than MIN_GROUP_N
    are dropped. strata: pd.Series of the same index, or None (free permutations only).
    Returns a dict, or None when fewer than 2 groups remain."""
    grp  = grouping.reindex(dm.ids).dropna()
    grp  = grp[grp.map(grp.value_counts()) >= MIN_GROUP_N]
    if grp.nunique() < 2:
        return None
    ids  = grp.index.tolist()
    dm_s = dm.filter(ids)
    labels = pd.factorize(grp.values)[0]
    n, g   = len(labels), labels.max() + 1

    # PERMANOVA: skbio (observed F, free p) + restricted p
    res_pm = permanova(dm_s, grp.values, permutations=N_PERM, seed=SEED)
    f_pm   = res_pm['test statistic']
    d2     = dm_s.data ** 2
    assert np.isclose(pseudo_f(d2, labels), f_pm), 'helper pseudo-F differs from skbio'

    # PERMDISP: PCoA of the tested flowers (as skbio does for a DistanceMatrix input).
    # Distances to centroid are taken before calling skbio: permdisp adds a
    # 'grouping' column to ordination.samples, so an ordination passed to it must
    # not be used again.
    ord_s  = pcoa(dm_s)
    z      = centroid_distances(ord_s, labels)
    z2     = (z[:, None] - z[None, :]) ** 2
    res_pd = permdisp(ord_s, grp.values, test='centroid', permutations=N_PERM, seed=SEED)
    f_pd   = res_pd['test statistic']
    assert np.isclose(pseudo_f(z2, labels), f_pd), 'helper PERMDISP F differs from skbio'

    out = {
        'n': n, 'n_groups': g, 'n_perm': N_PERM,
        'groups': '; '.join(f'{k} {v}' for k, v in grp.value_counts().sort_index().items()),
        'permanova_F': f_pm, 'R2': r2_from_f(f_pm, n, g),
        'permanova_p_free': res_pm['p-value'],
        'permdisp_F': f_pd, 'permdisp_p_free': res_pd['p-value'],
        'permanova_p_restr': np.nan, 'permdisp_p_restr': np.nan,
    }
    if strata is not None:
        st = strata.reindex(ids).values
        out['permanova_p_restr'] = restricted_p(d2, labels, st, f_pm, N_PERM, SEED)
        out['permdisp_p_restr']  = restricted_p(z2, labels, st, f_pd, N_PERM, SEED)
    return out

## Test plan

| block | variables | flowers | restricted permutations |
|---|---|---|---|
| design | sample_type | all | within tree (as 10a) |
| design | farm_id, management_type, collection_batch, tree_id | all | none: farm-level, R² descriptive |
| phenotype | pollination, germination, hyphae, spores, fungal colonization | measured; unbagged only | within farm x treatment |
| batch | extraction_batch, PCR / Qubit batch, primers, PCR redo | all | within farm |

Phenotypes are permuted within farm x treatment because most of them follow the
treatment (pollen reaches mainly unbagged flowers); in the unbagged subset this is
within farm. Batch columns are permuted within farm because PCR redo and the batches
are partly aligned with farm (01c). Within a farm, extraction batch and ITS1 PCR batch
equal the treatment (analysis_log), so their restricted test is a treatment test.

BH correction within each marker x metric x block.

In [ ]:
PHENOTYPES = ['is_pollination_clsm', 'pollination_class', 'is_germination',
              'is_hyphae_present', 'is_spore_present', 'fungal_colonization_score',
              'fungal_col_0v5']
# '{m}' is replaced by the marker suffix of the metadata columns (16s / its1)
BATCH = ['extraction_batch', 'pcr_batch_{m}', 'qubit_batch_{m}',
         'fwd_primer_{m}', 'rev_primer_{m}', 'pcr_redo_{m}_any']

# (variable, block, subset, strata column or None)
TESTS = (
    [('sample_type', 'design', 'all', 'tree_id')]
    + [(v, 'design', 'all', None)
       for v in ['farm_id', 'management_type', 'collection_batch', 'tree_id']]
    + [(v, 'phenotype', s, 'farm_treatment') for v in PHENOTYPES for s in ['measured', 'unbagged']]
    + [(v, 'batch', 'all', 'farm_id') for v in BATCH]
)
SUBSETS = {
    'all':      meta.index,
    'measured': meta.index,   # flowers without a value are dropped in run_test
    'unbagged': meta.index[meta['sample_type'] == 'unbagged_flower'],
}
print(f'{len(TESTS)} tests x {len(MARKERS)} markers x {len(METRICS)} metrics')

In [ ]:
def screen_all():
    """Run every test of TESTS for both markers and all metrics; add BH-adjusted p."""
    rows = []
    for marker in MARKERS:
        for metric in METRICS:
            print(f'{marker} {metric} ...')
            for variable, block, subset, strata in TESTS:
                column = variable.format(m=MARKER_COL[marker])
                res = run_test(
                    dist[marker][metric],
                    grouping=meta.loc[SUBSETS[subset], column],
                    strata=meta[strata] if strata else None,
                )
                if res is None:
                    print(f'  skip {column} ({subset}): < 2 groups of >= {MIN_GROUP_N}')
                    continue
                rows.append({'marker': marker, 'metric': metric, 'block': block,
                             'variable': variable.replace('_{m}', ''), 'column': column,
                             'subset': subset, 'strata': strata or 'none', **res})
    results = pd.DataFrame(rows)

    # Main p: restricted where available, free otherwise. BH within
    # marker x metric x block, separately for PERMANOVA and PERMDISP.
    for test in ['permanova', 'permdisp']:
        results[f'{test}_p'] = results[f'{test}_p_restr'].fillna(results[f'{test}_p_free'])
        results[f'{test}_p_adj'] = (
            results.groupby(['marker', 'metric', 'block'])[f'{test}_p']
            .transform(lambda p: multipletests(p, method='fdr_bh')[1])
        )
    return results


results = cached(TAB_DIR / f'05_permanova_screen{FIG_SUFFIX}.csv', screen_all)

results.query("metric == 'aitchison_CLR'")[
    ['marker', 'block', 'variable', 'subset', 'n', 'R2', 'permanova_p_free',
     'permanova_p_restr', 'permanova_p_adj', 'permdisp_p_restr', 'permdisp_p_adj']
].round(3)

## Figure A: summary heatmap

Rows = variable (and subset), grouped by block; columns = metric; one panel per marker.
Fill = PERMANOVA R². Label = BH-adjusted p (restricted where available, free for the
farm-level design variables, marked †). * p_adj < ALPHA. Bold border = PERMDISP
p_adj < ALPHA: the groups also differ in spread, so part of R² may be dispersion.

In [ ]:
results['row_label'] = np.where(results['subset'] == 'all', results['variable'],
                                results['variable'] + ' (' + results['subset'] + ')')
row_order = results.drop_duplicates('row_label')['row_label'].tolist()   # order of TESTS
block_of  = results.drop_duplicates('row_label').set_index('row_label')['block']

# figsize: width per marker panel, height grows with the number of rows
fig, axes = plt.subplots(1, len(MARKERS), figsize=(6.5 * len(MARKERS), 0.32 * len(row_order) + 2),
                         sharey=True)
r2_max = results['R2'].max()   # one colour scale for both panels

for ax, marker in zip(axes, MARKERS):
    sub = results[results['marker'] == marker]
    r2  = sub.pivot(index='row_label', columns='metric', values='R2').reindex(index=row_order, columns=METRICS)
    pad = sub.pivot(index='row_label', columns='metric', values='permanova_p_adj').reindex_like(r2)
    pdp = sub.pivot(index='row_label', columns='metric', values='permdisp_p_adj').reindex_like(r2)
    free = sub.pivot(index='row_label', columns='metric', values='strata').reindex_like(r2) == 'none'

    labels = pad.map(lambda p: '' if pd.isna(p) else f'{p:.3f}' + (' *' if p < ALPHA else ''))
    labels = labels.where(~free, labels + ' †')
    # cmap: fill colour scale of R2; annot_kws size: font size of the p labels
    sns.heatmap(r2, ax=ax, cmap='YlOrRd', vmin=0, vmax=r2_max, annot=labels, fmt='',
                annot_kws={'size': 6}, linewidths=0.3, linecolor='grey',
                cbar=(marker == list(MARKERS)[-1]), cbar_kws={'label': 'R²'})
    # Bold border: PERMDISP significant (linewidth controls the border width)
    for i, row in enumerate(r2.index):
        for j, col in enumerate(r2.columns):
            if pdp.loc[row, col] < ALPHA:
                ax.add_patch(Rectangle((j, i), 1, 1, fill=False, edgecolor='black', lw=1.8))
    # Horizontal lines between blocks
    for i in range(1, len(row_order)):
        if block_of[row_order[i]] != block_of[row_order[i - 1]]:
            ax.axhline(i, color='black', lw=1.2)
    ax.set_title(f'{marker}', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', labelrotation=30, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)

fig.suptitle(f'PERMANOVA screen: R² (fill), BH-adjusted p (label); * p_adj < {ALPHA}; '
             f'bold border = PERMDISP p_adj < {ALPHA}; † free permutations ({N_PERM} permutations)',
             fontsize=10)
plt.tight_layout()
save_fig(fig, '05_permanova_screen_heatmap')
plt.show()

## Figures B and C: ordinations and dispersion of key variables (CLR Aitchison)

B: PCoA of all flowers (grey) with the flowers of the tested subset coloured by group;
large markers = group centroids. C: distance of each flower to its group centroid
(the PERMDISP data). Variables are chosen in `KEY_VARIABLES`.

In [ ]:
# (column, subset) pairs shown in figures B and C; '{m}' = marker suffix
KEY_VARIABLES = [('sample_type', 'all'), ('farm_id', 'all'),
                 ('pollination_class', 'unbagged'), ('is_hyphae_present', 'measured'),
                 ('fungal_col_0v5', 'unbagged'), ('extraction_batch', 'all'),
                 ('pcr_redo_{m}_any', 'all')]
METRIC_FIG = 'aitchison_CLR'


def key_groups(marker, column, subset):
    """Group labels of the flowers in one key panel (same rules as run_test)."""
    grp = meta.loc[SUBSETS[subset], column].reindex(dist[marker][METRIC_FIG].ids).dropna()
    return grp[grp.map(grp.value_counts()) >= MIN_GROUP_N]


def stats_title(marker, column, subset):
    r = results.query('marker == @marker and metric == @METRIC_FIG and column == @column '
                      'and subset == @subset').iloc[0]
    return (f"{column} ({subset}), n = {r['n']}\nR² {r['R2']:.3f}, p_adj {r['permanova_p_adj']:.3f}; "
            f"PERMDISP p_adj {r['permdisp_p_adj']:.3f}")


for marker in MARKERS:
    ordination = ords[marker][METRIC_FIG]
    coords     = ordination.samples.iloc[:, :2]
    ev         = ordination.proportion_explained.values[:2] * 100

    # Figure B: 3 panels per row; figsize scales with the number of rows
    ncol = 3
    nrow = int(np.ceil(len(KEY_VARIABLES) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 5 * nrow))
    for ax, (col_tmpl, subset) in zip(axes.flat, KEY_VARIABLES):
        column = col_tmpl.format(m=MARKER_COL[marker])
        grp    = key_groups(marker, column, subset)
        # grey background: every flower in the ordination
        ax.scatter(coords.iloc[:, 0], coords.iloc[:, 1], c='lightgrey', s=8, alpha=0.5)
        # palette: tab10 up to 10 groups, husl beyond (extraction and PCR batches)
        levels  = sorted(grp.unique())
        palette = dict(zip(levels, sns.color_palette('tab10' if len(levels) <= 10 else 'husl',
                                                     len(levels))))
        for lab in levels:
            xy = coords.loc[grp.index[grp == lab]]
            ax.scatter(xy.iloc[:, 0], xy.iloc[:, 1], color=palette[lab], s=14, alpha=0.75,
                       label=f'{lab} ({len(xy)})')
            # centroid: large marker with black edge
            ax.scatter(*xy.mean().values, color=palette[lab], s=140, marker='X',
                       edgecolor='black', linewidth=0.8)
        ax.set_title(stats_title(marker, column, subset), fontsize=8)
        ax.set_xlabel(f'PCo1 ({ev[0]:.1f}%)', fontsize=8)
        ax.set_ylabel(f'PCo2 ({ev[1]:.1f}%)', fontsize=8)
        # legend outside the panel would crowd the grid; small font inside instead
        ax.legend(fontsize=5, markerscale=0.8, ncol=2 if len(levels) > 6 else 1, loc='best')
    for ax in axes.flat[len(KEY_VARIABLES):]:
        ax.set_visible(False)
    fig.suptitle(f'{marker}: PCoA, {METRIC_FIG}; coloured = flowers of the tested subset, '
                 f'X = group centroid', fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'05_{marker}_pcoa_key_variables')
    plt.show()

    # Figure C: distance to centroid, one panel per key variable
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 4 * nrow))
    for ax, (col_tmpl, subset) in zip(axes.flat, KEY_VARIABLES):
        column = col_tmpl.format(m=MARKER_COL[marker])
        grp    = key_groups(marker, column, subset)
        dm_s   = dist[marker][METRIC_FIG].filter(grp.index.tolist())
        z      = centroid_distances(pcoa(dm_s), pd.factorize(grp.values)[0])
        df     = pd.DataFrame({'group': grp.values, 'dist_to_centroid': z})
        order  = sorted(grp.unique())
        # box: quartiles; strip: individual flowers (size = point size, jitter = spread)
        sns.boxplot(data=df, x='group', y='dist_to_centroid', order=order, ax=ax,
                    color='white', showfliers=False)
        sns.stripplot(data=df, x='group', y='dist_to_centroid', order=order, ax=ax,
                      size=2.5, alpha=0.6, jitter=0.25)
        ax.set_title(stats_title(marker, column, subset), fontsize=8)
        ax.set_xlabel('')
        ax.tick_params(axis='x', labelrotation=45, labelsize=7)
    for ax in axes.flat[len(KEY_VARIABLES):]:
        ax.set_visible(False)
    fig.suptitle(f'{marker}: PERMDISP, distance to group centroid ({METRIC_FIG})',
                 fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'05_{marker}_permdisp_key_variables')
    plt.show()

## Figure D: batch columns vs design (Cramér's V)

How strongly each batch column is tied to farm, treatment and farm x treatment, on the
flowers of this sample set (0 = independent, 1 = one determines the other). A batch
column with V near 1 cannot be separated from that design variable by any test.

In [ ]:
flowers  = meta.loc[sorted(set().union(*[dist[m]['aitchison_CLR'].ids for m in MARKERS]))]
batch_cols = (['collection_batch', 'extraction_batch']
              + [b.format(m=m) for m in MARKER_COL.values() for b in BATCH[1:]])
design_cols = ['farm_id', 'sample_type', 'farm_treatment']

cramers_v = pd.DataFrame(
    [[association(pd.crosstab(flowers[b], flowers[d]).values, method='cramer')
      for d in design_cols] for b in batch_cols],
    index=batch_cols, columns=design_cols,
)

# figsize: width for 3 columns, height per batch column; annot: V values
fig, ax = plt.subplots(figsize=(5, 0.4 * len(batch_cols) + 1.5))
sns.heatmap(cramers_v, annot=True, fmt='.2f', cmap='Greys', vmin=0, vmax=1,
            annot_kws={'size': 8}, cbar_kws={'label': "Cramér's V"}, ax=ax)
ax.set_title(f"Batch vs design, {len(flowers)} flowers", fontsize=10)
plt.tight_layout()
save_fig(fig, '05_batch_cramers_v')
plt.show()
cramers_v.round(2)

## Figure E: extraction batch 2024-09-20

The only extraction batch holding both treatments and two farms: mt (2 bagged,
3 unbagged) and yb (3 bagged, 1 unbagged). In CLR space, is each of these flowers
closer to its own farm x treatment group, or to the other flowers of the batch?
- `to own group`: distance to the centroid of its farm x treatment group, without the
  batch flowers;
- `to batch`: distance to the centroid of the other batch flowers;
- reference: the other flowers of the same groups, distance to their own group centroid
  (leave-one-out).
Then a descriptive PERMANOVA within mt and within yb with grouping = extraction batch
(free permutations; few flowers, low power).

In [ ]:
BATCH_ID = '2024-09-20'

batch_rows, within_farm = [], []
for marker in MARKERS:
    x     = clr_data[marker]                       # samples x features, Euclidean = Aitchison
    m     = meta.loc[x.index]
    in_b  = (m['extraction_batch'] == BATCH_ID).values
    for i in np.flatnonzero(in_b):
        sid, grp = x.index[i], m['farm_treatment'].iloc[i]
        own   = x[(m['farm_treatment'] == grp).values & ~in_b].mean()
        other = x[in_b & (x.index != sid)].mean()
        batch_rows += [
            {'marker': marker, 'sample': sid, 'group': grp, 'kind': 'batch flower: to own group',
             'distance': np.linalg.norm(x.loc[sid] - own)},
            {'marker': marker, 'sample': sid, 'group': grp, 'kind': 'batch flower: to batch',
             'distance': np.linalg.norm(x.loc[sid] - other)},
        ]
    # Reference: non-batch flowers of the same groups, leave-one-out group centroid
    for grp in m.loc[in_b, 'farm_treatment'].unique():
        xg = x[(m['farm_treatment'] == grp).values & ~in_b]
        n  = len(xg)
        loo = (n * xg.mean().values - xg.values) / (n - 1)
        for sid, d in zip(xg.index, np.linalg.norm(xg.values - loo, axis=1)):
            batch_rows.append({'marker': marker, 'sample': sid, 'group': grp,
                               'kind': 'reference: to own group', 'distance': d})
    # Descriptive PERMANOVA within mt and within yb
    for farm in ['mt', 'yb']:
        res = run_test(dist[marker]['aitchison_CLR'],
                       grouping=meta.loc[meta['farm_id'] == farm, 'extraction_batch'])
        within_farm.append({'marker': marker, 'farm': farm, **res})

batch_dist  = pd.DataFrame(batch_rows)
within_farm = pd.DataFrame(within_farm)
batch_dist.to_csv(TAB_DIR / f'05_batch_{BATCH_ID}_distances{FIG_SUFFIX}.csv', index=False)
within_farm.to_csv(TAB_DIR / f'05_batch_{BATCH_ID}_within_farm{FIG_SUFFIX}.csv', index=False)

# One panel per marker; strip of distances per kind, coloured by farm x treatment group
fig, axes = plt.subplots(1, len(MARKERS), figsize=(6 * len(MARKERS), 4.5))
kinds = ['reference: to own group', 'batch flower: to own group', 'batch flower: to batch']
for ax, marker in zip(axes, MARKERS):
    d = batch_dist[batch_dist['marker'] == marker]
    # reference flowers jittered (jitter = horizontal spread); batch flowers not
    # jittered, so the lines below meet their points
    sns.stripplot(data=d[d['kind'] == kinds[0]], x='kind', y='distance', hue='group',
                  order=kinds, jitter=0.2, size=5, alpha=0.8, ax=ax)
    sns.stripplot(data=d[d['kind'] != kinds[0]], x='kind', y='distance', hue='group',
                  order=kinds, jitter=False, size=5, alpha=0.8, ax=ax, legend=False)
    # lines join the two distances of each batch flower (to own group, to batch)
    pairs = (batch_dist[(batch_dist['marker'] == marker) & batch_dist['kind'].str.startswith('batch')]
             .pivot(index='sample', columns='kind', values='distance'))
    for _, r in pairs.iterrows():
        ax.plot([1, 2], [r[kinds[1]], r[kinds[2]]], color='grey', lw=0.6, zorder=0)
    ax.set_title(f'{marker}: CLR Aitchison distance to centroids', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=8)
    ax.legend(fontsize=7, title='farm x treatment', title_fontsize=7)
fig.suptitle(f'Extraction batch {BATCH_ID}: own farm x treatment group or batch?',
             fontweight='bold')
plt.tight_layout()
save_fig(fig, f'05_batch_{BATCH_ID}_distances')
plt.show()

within_farm[['marker', 'farm', 'n', 'groups', 'R2', 'permanova_p_free', 'permdisp_p_free']].round(3)

## Figure F: treatment effect per farm and the shared 16S PCR batch

In some farms the bagged and unbagged flowers share a 16S PCR batch (kk, sa, yb in the
288 set; found from the metadata below). There the 16S treatment contrast is free of
PCR batch (not of extraction batch). If the 16S treatment R² in these farms is similar
to the other farms, PCR batch adds little to it. ITS1 PCR batches are separate in every
farm and serve as the reference. Per-farm PERMANOVA of sample_type, permutations within
tree (as the per-farm test of 10a).

In [ ]:
# Farms whose 16S PCR batches hold both treatments
shared = (meta.groupby(['farm_id', 'pcr_batch_16s'])['sample_type'].nunique() > 1)
SHARED_16S_PCR = sorted(shared[shared].index.get_level_values('farm_id').unique())
print('Farms with bagged and unbagged in one 16S PCR batch:', SHARED_16S_PCR)

def treatment_per_farm():
    """sample_type PERMANOVA / PERMDISP within each farm, permutations within tree."""
    rows = []
    for marker in MARKERS:
        for metric in METRICS:
            for farm in FARM_LEVELS:
                res = run_test(dist[marker][metric],
                               grouping=meta.loc[meta['farm_id'] == farm, 'sample_type'],
                               strata=meta['tree_id'])
                if res is not None:
                    rows.append({'marker': marker, 'metric': metric, 'farm': farm, **res})
    return pd.DataFrame(rows)


per_farm = cached(TAB_DIR / f'05_treatment_per_farm{FIG_SUFFIX}.csv', treatment_per_farm)
per_farm['shared_16S_pcr'] = per_farm['farm'].isin(SHARED_16S_PCR)

# One panel per metric; x = farm, y = R2, colour = marker;
# filled marker = restricted p < ALPHA, open = not significant;
# grey band = farms with a shared 16S PCR batch
MARKER_COLOURS = {'16S': '#4C72B0', 'ITS1': '#C44E52'}
fig, axes = plt.subplots(1, len(METRICS), figsize=(4.5 * len(METRICS), 4), sharey=True)
for ax, metric in zip(axes, METRICS):
    for j, farm in enumerate(FARM_LEVELS):
        if farm in SHARED_16S_PCR:
            ax.axvspan(j - 0.45, j + 0.45, color='grey', alpha=0.15, lw=0)
    for k, marker in enumerate(MARKERS):
        d = per_farm[(per_farm['metric'] == metric) & (per_farm['marker'] == marker)]
        xpos = [FARM_LEVELS.index(f) + (k - 0.5) * 0.25 for f in d['farm']]   # dodge markers
        sig  = (d['permanova_p_restr'] < ALPHA).values
        ax.scatter(np.array(xpos)[sig], d['R2'].values[sig], color=MARKER_COLOURS[marker],
                   s=40, label=marker if metric == METRICS[0] else None)
        ax.scatter(np.array(xpos)[~sig], d['R2'].values[~sig], facecolor='white',
                   edgecolor=MARKER_COLOURS[marker], s=40)
    ax.set_xticks(range(len(FARM_LEVELS)), FARM_LEVELS)
    ax.set_title(metric, fontsize=10)
    ax.set_xlabel('farm')
axes[0].set_ylabel('R² of sample_type (within farm)')
axes[0].legend(fontsize=8)
fig.suptitle(f'Treatment effect per farm; filled = p < {ALPHA} (permutations within tree); '
             f'grey = bagged and unbagged in one 16S PCR batch ({", ".join(SHARED_16S_PCR)})',
             fontsize=10)
plt.tight_layout()
save_fig(fig, '05_treatment_per_farm')
plt.show()

per_farm.query("metric == 'aitchison_CLR'").pivot(index='farm', columns='marker', values='R2').reindex(FARM_LEVELS).round(3)

## Combined PDF

All figures of this notebook in one PDF, for viewing. The PNG files are the
per-figure output.

In [ ]:
# One PDF with every figure of this notebook, in the order they were saved.
# Run the whole notebook before this cell.
from matplotlib.backends.backend_pdf import PdfPages

pdf_path = FIG_DIR / f'05_16S_ITS1_permanova_screen{FIG_SUFFIX}.pdf'
with PdfPages(pdf_path) as pdf:
    for fig in FIGURES.values():
        pdf.savefig(fig, bbox_inches='tight')
print(f'{len(FIGURES)} figures -> {pdf_path}')